# 05_catboost

Этот ноутбук посвящен модели CatBoost как наиболее сильному нелинейному табличному подходу
в текущем наборе воспроизводимых экспериментов.
Цель этапа — получить сопоставимые результаты на тех же временных сплитах,
что и для `mean baseline` и `ridge`, а затем построить финальный прогноз для Kaggle.

Метрики сохраняются в двух постановках:
- `all_series` — для прямого сравнения с другими глобальными tabular-моделями;
- `top_pairs` — как отдельная практическая постановка и как сопоставимый ориентир
  с ранее полученным результатом SARIMA на этом подмножестве.


## Методологическая оговорка

Полный подбор гиперпараметров CatBoost на всех временных сплитах слишком дорог вычислительно,
поэтому здесь используется более практичная схема.
Гиперпараметры либо загружаются из заранее сохраненного JSON-файла,
либо, при явном включении `RUN_OPTUNA`, подбираются на одном proxy-сплите.

Итоговые результаты CatBoost на `all_series` и `top_pairs`
следует интерпретировать как оценку фиксированной рабочей конфигурации модели,
а не как отдельно оптимизированные под каждую постановку гиперпараметры.


In [1]:
import os
import json
from pathlib import Path
from datetime import datetime
import time

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor, Pool
import optuna

import warnings
warnings.filterwarnings("ignore")

# Настройка pandas
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Логгер
import logging

logger = logging.getLogger("catboost_store_level")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger catboost_store_level initialized")


2026-04-19 18:21:25,831 - catboost_store_level - INFO - Logger catboost_store_level initialized


In [2]:
PROJECT_ROOT = Path('..').resolve()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DATASETS_DIR = ARTIFACTS_DIR / 'datasets'
METADATA_DIR = ARTIFACTS_DIR / 'metadata'
METRICS_DIR = ARTIFACTS_DIR / 'metrics'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'
PARAMS_DIR = ARTIFACTS_DIR / 'params'
SUBMISSIONS_DIR = ARTIFACTS_DIR / 'submissions'

for path in [METRICS_DIR, PREDICTIONS_DIR, PARAMS_DIR, SUBMISSIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

train_processed = pd.read_csv(DATASETS_DIR / 'train_processed.csv', parse_dates=['date'])
test_processed = pd.read_csv(DATASETS_DIR / 'test_processed.csv', parse_dates=['date'])
top_pairs_df = pd.read_csv(DATASETS_DIR / 'top_pairs.csv')
with open(METADATA_DIR / 'splits.json', 'r', encoding='utf-8') as f:
    splits_json = json.load(f)

splits = []
for s in splits_json:
    splits.append({
        'name': s['name'],
        'train_idx': pd.Index(s['train_idx']),
        'val_idx': pd.Index(s['val_idx']),
        'train_end': pd.to_datetime(s['train_end']),
        'val_start': pd.to_datetime(s['val_start']),
        'val_end': pd.to_datetime(s['val_end']),
    })

top_pairs = set(top_pairs_df[['store_nbr', 'family']].itertuples(index=False, name=None))
logger.info(f'Loaded train_processed: {train_processed.shape}')
logger.info(f'Loaded test_processed: {test_processed.shape}')
logger.info(f'Loaded top_pairs: {len(top_pairs)}')


2026-04-19 18:21:43,531 - catboost_store_level - INFO - Loaded train_processed: (3008016, 19)
2026-04-19 18:21:43,532 - catboost_store_level - INFO - Loaded test_processed: (28512, 18)
2026-04-19 18:21:43,533 - catboost_store_level - INFO - Loaded top_pairs: 357


## Политика признаков

Для CatBoost особенно важно заранее договориться о наборе признаков.
С одной стороны, хочется использовать богатую табличную структуру.
С другой стороны, нельзя тащить в модель признаки, которые дают утечку
или плохо согласуются с будущим прогнозным горизонтом.


In [3]:
def add_time_features(df):
    """
    Добавляет базовые временные признаки в df:
    - dayofweek, weekofyear, month, day, is_weekend.
    Ожидает колонку 'date' типа datetime64[ns].
    """
    df = df.copy()
    df["dow"] = df["date"].dt.weekday.astype("int8")
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype("int16")
    df["month"] = df["date"].dt.month.astype("int8")
    df["day"] = df["date"].dt.day.astype("int8")
    df["is_weekend"] = df["dow"].isin([5, 6]).astype("int8")
    return df


In [4]:
def add_lag_features(df, lags, group_cols=["store_nbr", "family"], target_col="sales"):
    """
    Добавляет лаги target_col на заданные сдвиги в днях.
    """
    df = df.sort_values(["date"] + group_cols).copy()
    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = (
            df.groupby(group_cols)[target_col]
            .shift(lag)
        )
    return df

def add_rolling_features(df, windows, group_cols=["store_nbr", "family"], target_col="sales"):
    df = df.sort_values(["date"] + group_cols).copy()
    g = df.groupby(group_cols)[target_col]
    for w in windows:
        df[f"{target_col}_rolling_mean_{w}"] = (
            g.shift(1).rolling(window=w, min_periods=1).mean()
        )
    return df


In [5]:
# Список категориальных признаков для CatBoost
CAT_FEATURES = [
    "store_nbr",
    "family",
    "city",
    "state",
    "store_type",
    "cluster",
    "type",
    "locale",
    "agg_description",
]

# Бинарные/индикаторные признаки
BIN_FEATURES = [
    "is_holiday",
    "is_payday",
    "transferred",
]

# Базовые безопасные численные признаки для общего и submission-режима
SAFE_NUM_FEATURES_BASE = [
    "onpromotion",
]

# Эти признаки можно возвращать только после отдельной проверки доступности
# на тестовом горизонте и отсутствия методологических проблем.
OPTIONAL_RESEARCH_NUM_FEATURES = [
    col for col in ["transactions", "dcoilwtico"]
    if col in train_processed.columns and col in test_processed.columns
]

NUM_FEATURES_BASE = SAFE_NUM_FEATURES_BASE + OPTIONAL_RESEARCH_NUM_FEATURES

# Временные фичи, которые добавим
TIME_FEATURES = ["dow", "weekofyear", "month", "day", "is_weekend"]

# Runtime-переключатели для CatBoost
CATBOOST_TASK_TYPE = "GPU"  # change to "GPU" in cloud runtime
CATBOOST_DEVICES = "0"
RUN_OPTUNA = False


In [6]:
def format_seconds(seconds):
    """
    Форматирует длительность в секундах в компактный человекочитаемый вид.
    """
    if seconds is None or not np.isfinite(seconds):
        return "unknown"

    seconds = int(max(seconds, 0))
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}h {minutes:02d}m {secs:02d}s"
    if minutes > 0:
        return f"{minutes}m {secs:02d}s"
    return f"{secs}s"

class ProgressTimer:
    """
    Простой ETA-таймер по среднему времени уже завершённых шагов.
    """
    def __init__(self, total_steps, label):
        self.total_steps = max(int(total_steps), 1)
        self.label = label
        self.started_at = time.perf_counter()
        self.completed_steps = 0

    def log_step(self, logger, step_name):
        self.completed_steps += 1
        elapsed = time.perf_counter() - self.started_at
        avg_step = elapsed / self.completed_steps
        remaining_steps = max(self.total_steps - self.completed_steps, 0)
        eta_seconds = avg_step * remaining_steps

        logger.info(
            f"{self.label} | done {self.completed_steps}/{self.total_steps} ({step_name}) | "
            f"elapsed={format_seconds(elapsed)} | eta={format_seconds(eta_seconds)}"
        )

def fill_na_in_cat(df, feature_cols, cat_features_idx):
    """
    Заполняет NaN в категориальных колонках специальным токеном "__MISSING__".
    Это нужно, т.к. CatBoost не принимает NaN в cat_features.
    """
    df = df.copy()
    cat_cols = [feature_cols[i] for i in cat_features_idx]
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype("object").fillna("__MISSING__")
    return df

def filter_pairs_by_scope(df, top_pairs, use_top_pairs):
    if not use_top_pairs:
        return df.copy()
    mask = df[["store_nbr", "family"]].apply(tuple, axis=1).isin(top_pairs)
    return df[mask].copy()

def make_train_val_for_split(
    split,
    full_df,
    top_pairs,
    use_top_pairs=True,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Готовит X_train, y_train, X_val, y_val для заданного сплита.

    Параметры:
    - split: словарь из splits (name, train_idx, val_idx, ...)
    - full_df: train_processed
    - top_pairs: множество (store_nbr, family)
    - use_top_pairs: если True, обучаемся и валидируемся только на top_pairs
    - lags: tuple лагов для sales
    - roll_windows: окна для скользящего среднего по sales

    Возвращает:
    - X_train, y_train, X_val, y_val, cat_features
    """
    split_name = split["name"]
    scope_name = "top_pairs" if use_top_pairs else "all_series"
    logger.info(f"Preparing data for {split_name} | scope={scope_name}")

    # Берём train/val по индексам
    train_df = full_df.loc[split["train_idx"]].copy()
    val_df   = full_df.loc[split["val_idx"]].copy()

    train_df = filter_pairs_by_scope(train_df, top_pairs, use_top_pairs)
    val_df   = filter_pairs_by_scope(val_df, top_pairs, use_top_pairs)

    logger.info(
        f"{split_name} | scope={scope_name} | train rows after filtering: {len(train_df)}, "
        f"val rows: {len(val_df)}"
    )

    # Склеиваем train+val для корректного расчёта лагов (история перед валидацией)
    combined = pd.concat([train_df, val_df], axis=0).sort_values(["store_nbr", "family", "date"])

    # Добавляем временные фичи
    combined = add_time_features(combined)

    # Лаги по sales
    combined = add_lag_features(combined, lags=lags, group_cols=["store_nbr", "family"], target_col="sales")

    # Скользящие средние
    combined = add_rolling_features(combined, windows=roll_windows, group_cols=["store_nbr", "family"], target_col="sales")

    # После лагов/rolling первые дни ряда будут с NaN — отрежем их
    # Но делаем это аккуратно: только там, где NaN в хотя бы одном из lag/rolling.
    feature_cols_lag_roll = [col for col in combined.columns if "lag_" in col or "rolling_mean_" in col]
    combined = combined.dropna(subset=feature_cols_lag_roll)

    # Разделяем обратно на train/val по дате
    train_end = train_df["date"].max()
    val_start = val_df["date"].min()
    val_end   = val_df["date"].max()

    train_mask = (combined["date"] <= train_end)
    val_mask   = (combined["date"] >= val_start) & (combined["date"] <= val_end)

    train_fe = combined[train_mask].copy()
    val_fe   = combined[val_mask].copy()

    logger.info(
        f"{split_name} | after lag/rolling dropna: train={len(train_fe)}, val={len(val_fe)}"
    )

    # Целевая переменная
    y_train = train_fe["sales"].astype("float32")
    y_val   = val_fe["sales"].astype("float32")

    # Формируем список всех фичей
    feature_cols = (
        NUM_FEATURES_BASE
        + TIME_FEATURES
        + feature_cols_lag_roll
        + BIN_FEATURES
        + CAT_FEATURES
    )

    # Оставляем только существующие в датафрейме
    feature_cols = [c for c in feature_cols if c in train_fe.columns]

    X_train = train_fe[feature_cols].copy()
    X_val   = val_fe[feature_cols].copy()

    # CatBoost требует индексы категориальных колонок
    cat_features = [i for i, c in enumerate(feature_cols) if c in CAT_FEATURES + BIN_FEATURES]

    # CatBoost не принимает NaN в категориальных признаках.
    X_train = fill_na_in_cat(X_train, feature_cols, cat_features)
    X_val = fill_na_in_cat(X_val, feature_cols, cat_features)

    logger.info(
        f"{split_name} | X_train shape={X_train.shape}, X_val shape={X_val.shape}, "
        f"n_cat_features={len(cat_features)}"
    )

    return X_train, y_train, X_val, y_val, feature_cols, cat_features


In [7]:
def rmsle_metric(y_true, y_pred):
    """
    RMSLE в исходном масштабе (после обратного expm1).
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))


In [8]:
optuna_split = splits[0]
optuna_X_train, optuna_y_train, optuna_X_val, optuna_y_val, optuna_feature_cols, optuna_cat_features = make_train_val_for_split(
    optuna_split,
    train_processed,
    top_pairs,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
)
logger.info(f'Optuna proxy split: {optuna_split["name"]}')
logger.info(f'Optuna train shape: {optuna_X_train.shape}, val shape: {optuna_X_val.shape}')


2026-04-19 18:21:43,673 - catboost_store_level - INFO - Preparing data for split_1 | scope=top_pairs
2026-04-19 18:22:05,319 - catboost_store_level - INFO - split_1 | scope=top_pairs | train rows after filtering: 596904, val rows: 5712
2026-04-19 18:22:07,259 - catboost_store_level - INFO - split_1 | after lag/rolling dropna: train=586908, val=5712
2026-04-19 18:22:08,353 - catboost_store_level - INFO - split_1 | X_train shape=(586908, 25), X_val shape=(5712, 25), n_cat_features=12
2026-04-19 18:22:08,389 - catboost_store_level - INFO - Optuna proxy split: split_1
2026-04-19 18:22:08,390 - catboost_store_level - INFO - Optuna train shape: (586908, 25), val shape: (5712, 25)


In [9]:
def objective(trial):
    param = {
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 5.0),
        'border_count': trial.suggest_int('border_count', 64, 255),
    }

    y_train_log = np.log1p(optuna_y_train)
    y_val_log = np.log1p(optuna_y_val)

    optuna_X_train_filled = fill_na_in_cat(optuna_X_train, optuna_feature_cols, optuna_cat_features)
    optuna_X_val_filled = fill_na_in_cat(optuna_X_val, optuna_feature_cols, optuna_cat_features)

    train_pool = Pool(data=optuna_X_train_filled, label=y_train_log, cat_features=optuna_cat_features)
    val_pool = Pool(data=optuna_X_val_filled, label=y_val_log, cat_features=optuna_cat_features)

    model = CatBoostRegressor(
        loss_function='RMSE',
        eval_metric='RMSE',
        depth=param['depth'],
        learning_rate=param['learning_rate'],
        l2_leaf_reg=param['l2_leaf_reg'],
        bagging_temperature=param['bagging_temperature'],
        border_count=param['border_count'],
        random_state=42,
        thread_count=4,
        task_type=CATBOOST_TASK_TYPE,
        devices=CATBOOST_DEVICES if CATBOOST_TASK_TYPE == 'GPU' else None,
        verbose=False,
        od_type='Iter',
        od_wait=50,
        iterations=3000,
    )

    model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=False)
    y_pred_log = model.predict(val_pool)
    y_pred = np.expm1(y_pred_log)
    score = rmsle_metric(optuna_y_val, y_pred)
    logger.info(f'Trial {trial.number}: params={param}, RMSLE={score:.4f}')
    return score


## Подбор или загрузка гиперпараметров

На этом шаге используются либо заранее сохраненные гиперпараметры,
либо, при необходимости, запускается новый Optuna-поиск.
Для воспроизводимого и быстрого прогона ноутбука предпочтителен первый режим:
он позволяет не повторять дорогой подбор и использовать уже зафиксированную рабочую конфигурацию.


In [10]:
import multiprocessing

n_cores = multiprocessing.cpu_count()
logger.info(f"CPU cores available: {n_cores}")

N_TRIALS = 20  # можно увеличить/уменьшить по ситуации
OPTUNA_N_JOBS = 1 if CATBOOST_TASK_TYPE == "GPU" else 2
params_path = PARAMS_DIR / 'catboost_best_params.json'
legacy_params_path = PROJECT_ROOT.parent / 'data' / 'experiment_info' / 'catboost_best_params.json'

if RUN_OPTUNA:
    study = optuna.create_study(
        direction="minimize",
        study_name="catboost_rmsle_log1p",
    )

    logger.info(
        f"Starting Optuna optimization with {N_TRIALS} trials... "
        f"task_type={CATBOOST_TASK_TYPE}, optuna_n_jobs={OPTUNA_N_JOBS}"
    )

    optuna_eta_timer = ProgressTimer(total_steps=N_TRIALS, label="Optuna")

    def log_optuna_eta(study, trial):
        status = str(trial.state).split(".")[-1]
        best_suffix = ""
        complete_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        if complete_trials:
            best_suffix = f", best={study.best_value:.4f}"
        optuna_eta_timer.log_step(
            logger,
            f"trial {trial.number + 1}/{N_TRIALS}, state={status}{best_suffix}",
        )

    study.optimize(
        objective,
        n_trials=N_TRIALS,
        n_jobs=OPTUNA_N_JOBS,
        show_progress_bar=True,
        callbacks=[log_optuna_eta],
    )

    logger.info(f"Best RMSLE: {study.best_value:.4f}")
    logger.info(f"Best params: {study.best_params}")
else:
    study = None
    logger.info("RUN_OPTUNA=False: hyperparameter search skipped")


2026-04-19 18:22:08,441 - catboost_store_level - INFO - CPU cores available: 8
2026-04-19 18:22:08,443 - catboost_store_level - INFO - RUN_OPTUNA=False: hyperparameter search skipped


In [11]:
if RUN_OPTUNA:
    best_params = study.best_params
    best_params['n_trials'] = N_TRIALS
    best_params['best_value'] = study.best_value
    with open(params_path, 'w', encoding='utf-8') as f:
        json.dump(best_params, f, indent=2, ensure_ascii=False)
    logger.info(f'Best CatBoost params saved to {params_path}')
elif params_path.exists():
    with open(params_path, 'r', encoding='utf-8') as f:
        best_params = json.load(f)
    logger.info(f'Loaded CatBoost params from {params_path}')
elif legacy_params_path.exists():
    with open(legacy_params_path, 'r', encoding='utf-8') as f:
        best_params = json.load(f)
    logger.info(f'Loaded CatBoost params from legacy path {legacy_params_path}')
else:
    raise FileNotFoundError(
        'CatBoost params were not found. Either run Optuna or place catboost_best_params.json into artifacts/params or data/experiment_info.'
    )

best_params


2026-04-19 18:22:08,474 - catboost_store_level - INFO - Loaded CatBoost params from /home/jupyter/project/demand_forecasting/artifacts/params/catboost_best_params.json


{'depth': 6,
 'learning_rate': 0.27605587718305347,
 'l2_leaf_reg': 7.323277403098156,
 'bagging_temperature': 1.6861617510336562,
 'border_count': 71,
 'n_trials': 20,
 'best_value': 0.16994262483422473}

## Оценка на временных сплитах

Это основной блок ноутбука с точки зрения сопоставимого сравнения моделей.
CatBoost оценивается на тех же временных сплитах, что и более простые baseline-подходы,
причем результаты сохраняются отдельно для `all_series` и `top_pairs`.
Такая схема позволяет корректно сравнивать CatBoost с `mean baseline` и `ridge`
в глобальной постановке, а `top_pairs` использовать как дополнительный практический срез
и как сопоставимый ориентир с ранее полученным результатом SARIMA.


In [12]:
def calculate_metrics_all(y_true, y_pred):
    """
    Возвращает dict с RMSLE, RMSE, MAE, WAPE.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    # Обрезаем прогнозы по нулю
    y_pred = np.clip(y_pred, 0, None)

    # RMSLE
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

    # RMSE
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    # MAE
    mae = np.mean(np.abs(y_true - y_pred))

    # WAPE
    denom = np.sum(np.abs(y_true))
    wape = np.nan if denom == 0 else np.sum(np.abs(y_true - y_pred)) / denom

    return {
        "RMSLE": rmsle,
        "RMSE": rmse,
        "MAE": mae,
        "WAPE": wape,
    }


In [13]:
def fit_catboost_on_split(
    split,
    full_df,
    top_pairs,
    best_params,
    use_top_pairs=True,
    thread_count=4,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Обучает CatBoost на train-части сплита.
    Ничего не знает про валидацию и метрики — только fit.

    Возвращает:
    - model: обученный CatBoostRegressor
    - feature_cols: список фичей в том порядке, как их ждёт модель
    - cat_features_idx: индексы категориальных фичей в feature_cols
    """
    split_name = split["name"]
    scope_name = "top_pairs" if use_top_pairs else "all_series"
    logger.info(f"[{split_name}] Fitting CatBoost (train only) | scope={scope_name}")

    # Берём старую утилиту, но будем использовать только train-часть
    X_train, y_train, X_val_dummy, y_val_dummy, feature_cols, cat_features_idx = make_train_val_for_split(
        split,
        full_df,
        top_pairs,
        use_top_pairs=use_top_pairs,
        lags=lags,
        roll_windows=roll_windows,
    )

    # Лог-преобразование таргета
    y_train_log = np.log1p(y_train)

    train_pool = Pool(
        data=X_train,
        label=y_train_log,
        cat_features=cat_features_idx,
    )

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        depth=best_params["depth"],
        learning_rate=best_params["learning_rate"],
        l2_leaf_reg=best_params["l2_leaf_reg"],
        bagging_temperature=best_params["bagging_temperature"],
        border_count=best_params["border_count"],
        random_state=42,
        thread_count=thread_count,
        task_type=CATBOOST_TASK_TYPE,
        devices=CATBOOST_DEVICES if CATBOOST_TASK_TYPE == "GPU" else None,
        verbose=False,
        od_type="Iter",
        od_wait=50,
        iterations=2000,  # можно потом уменьшить
    )

    start_time = datetime.now()
    logger.info(f"[{split_name}] CatBoost fit started at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

    model.fit(
        train_pool,
        use_best_model=False,  # в этой функции только train, без val
        verbose=False,
    )

    end_time = datetime.now()
    elapsed = (end_time - start_time).total_seconds()
    logger.info(f"[{split_name}] CatBoost fit finished in {elapsed:.1f} seconds")

    return model, feature_cols, cat_features_idx


In [14]:
def prepare_history_and_future(split, full_df, top_pairs, use_top_pairs=True):
    """
    Готовит:
    - history_df: train-часть сплита, при необходимости отфильтрованную до top_pairs;
    - future_df: val-часть сплита, при необходимости отфильтрованную до top_pairs;
    - y_true: вектор истинных продаж для future_df (в том же порядке).
    """
    split_name = split["name"]
    scope_name = "top_pairs" if use_top_pairs else "all_series"

    train_df = full_df.loc[split["train_idx"]].copy()
    val_df   = full_df.loc[split["val_idx"]].copy()

    train_df = filter_pairs_by_scope(train_df, top_pairs, use_top_pairs)
    val_df   = filter_pairs_by_scope(val_df, top_pairs, use_top_pairs)

    logger.info(
        f"[{split_name}] history rows (train, scope={scope_name})={len(train_df)}, "
        f"future rows (val, scope={scope_name})={len(val_df)}"
    )

    history_df = train_df.sort_values(["store_nbr", "family", "date"]).copy()
    future_df  = val_df.sort_values(["store_nbr", "family", "date"]).copy()

    y_true = future_df["sales"].astype("float32").values

    # В future_df 'sales' будем перезаписывать предсказаниями, поэтому сохраним y_true отдельно.
    future_df = future_df.drop(columns=["sales"])

    return history_df, future_df, y_true


In [15]:
def recursive_forecast_catboost_on_split(
    split,
    full_df,
    top_pairs,
    model,
    feature_cols,
    cat_features_idx,
    use_top_pairs=True,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Делает рекурсивный прогноз CatBoost на 16-дневном val-окне сплита.

    Шаги:
    - берём history_df (train) и future_df (val) из prepare_history_and_future;
    - для каждого дня val-окна:
        * добавляем его «каркас» (без sales) к истории;
        * пересчитываем временные, лаговые и rolling-фичи;
        * предсказываем sales на этот день;
        * записываем предсказания в историю, чтобы они участвовали в лаговых фичах следующего дня.
    Возвращает:
    - val_pred_df: DataFrame с колонками [date, store_nbr, family, y_true, y_pred]
    """
    split_name = split["name"]
    scope_name = "top_pairs" if use_top_pairs else "all_series"
    logger.info(f"[{split_name}] Recursive CatBoost forecast on validation window | scope={scope_name}")

    history_df, future_df, y_true = prepare_history_and_future(split, full_df, top_pairs, use_top_pairs=use_top_pairs)

    # Будем накапливать историю с предсказаниями
    hist = history_df.copy()

    # Список предсказаний по дням
    preds_frames = []

    # Уникальные даты валидации по возрастанию
    val_dates = sorted(future_df["date"].unique())
    forecast_timer = ProgressTimer(total_steps=len(val_dates), label=f"[{split_name}] Validation forecast")

    for cur_date in val_dates:
        cur_ts = pd.to_datetime(cur_date)
        logger.info(f"[{split_name}] Forecasting date {cur_ts.date()}")

        # Строки для текущего дня (exogenous features, без sales)
        day_future = future_df[future_df["date"] == cur_date].copy()

        # Временно добавляем пустой sales (NaN), чтобы лаги считались от hist
        day_future["sales"] = np.nan

        # Склеиваем историю + текущий день
        combined = pd.concat([hist, day_future], axis=0, ignore_index=True)

        # Фичи: время, лаги, rolling
        combined = add_time_features(combined)
        combined = add_lag_features(
            combined,
            lags=lags,
            group_cols=["store_nbr", "family"],
            target_col="sales",
        )
        combined = add_rolling_features(
            combined,
            windows=roll_windows,
            group_cols=["store_nbr", "family"],
            target_col="sales",
        )

        # Берём только текущий день уже с фичами
        feature_cols_lag_roll = [c for c in combined.columns if "lag_" in c or "rolling_mean_" in c]
        cur_df = combined[combined["date"] == cur_date].copy()

        # Отбрасываем строки, где нет необходимых лагов/rolling
        cur_df = cur_df.dropna(subset=feature_cols_lag_roll)

        # Если вдруг всё отвалилось (нехватка истории) — пропускаем с предупреждением
        if cur_df.empty:
            logger.warning(f"[{split_name}] No rows to predict for {cur_ts.date()}, skipping")
            forecast_timer.log_step(logger, f"date={cur_ts.date()} skipped")
            continue

        # Готовим X для модели в том же порядке feature_cols
        X_cur = cur_df[feature_cols].copy()
        X_cur = fill_na_in_cat(X_cur, feature_cols, cat_features_idx)

        pool_cur = Pool(
            data=X_cur,
            cat_features=cat_features_idx,
        )

        # Предсказания в лог-пространстве → назад в исходный масштаб
        y_pred_log = model.predict(pool_cur)
        y_pred = np.expm1(y_pred_log)

        cur_df["y_pred"] = y_pred

        # Истинные продажи для этого дня берём из full_df
        true_day = full_df[full_df["date"] == cur_date].copy()
        true_day = filter_pairs_by_scope(true_day, top_pairs, use_top_pairs)
        true_day = true_day[["store_nbr", "family", "sales"]].rename(columns={"sales": "y_true"})

        cur_df = cur_df.merge(
            true_day,
            on=["store_nbr", "family"],
            how="left",
        )

        preds_frames.append(
            cur_df[["date", "store_nbr", "family", "y_true", "y_pred"]]
        )

        # Обновляем историю: для следующего дня 'sales' должны быть равны предсказанным y_pred
        # Берём ровно те строки, которые мы сейчас спрогнозировали (по ключам)
        hist_update = cur_df[["date", "store_nbr", "family"]].copy()
        hist_update["sales"] = y_pred

        # Также нужны остальные признаки (onpromotion, city, state, ...),
        # поэтому мёрджим их из future_df
        hist_update = hist_update.merge(
            future_df[future_df["date"] == cur_date].drop(columns=[]),
            on=["date", "store_nbr", "family"],
            how="left",
        )

        hist = pd.concat([hist, hist_update], axis=0, ignore_index=True)
        forecast_timer.log_step(logger, f"date={cur_ts.date()}")

    # Собираем всё вместе
    val_pred_df = pd.concat(preds_frames, axis=0, ignore_index=True)

    return val_pred_df


In [16]:
def evaluate_catboost_on_split_recursive(
    split,
    full_df,
    top_pairs,
    model,
    feature_cols,
    cat_features_idx,
    use_top_pairs=True,
    scope_name="top_pairs",
):
    """
    Обёртка: рекурсивный прогноз + расчёт метрик на сплите.
    Ответственность: только оценка качества.
    """
    split_name = split["name"]

    val_pred_df = recursive_forecast_catboost_on_split(
        split=split,
        full_df=full_df,
        top_pairs=top_pairs,
        use_top_pairs=use_top_pairs,
        model=model,
        feature_cols=feature_cols,
        cat_features_idx=cat_features_idx,
    )

    y_true = val_pred_df["y_true"].values
    y_pred = val_pred_df["y_pred"].values

    metrics = calculate_metrics_all(y_true, y_pred)
    metrics["split"] = split_name
    metrics["model"] = "catboost_store_family_recursive"
    metrics["scope"] = scope_name

    logger.info(
        f"[{split_name}] CatBoost (recursive) metrics: "
        f"RMSLE={metrics['RMSLE']:.4f}, "
        f"RMSE={metrics['RMSE']:.2f}, "
        f"MAE={metrics['MAE']:.2f}, "
        f"WAPE={metrics['WAPE']:.4f}"
    )

    return metrics, val_pred_df


In [17]:
catboost_metrics_results = {}
catboost_preds_results = {}

THREAD_COUNT = -1
scope_configs = [
    ("all_series", False),
    ("top_pairs", True),
]

for scope_name, use_top_pairs in scope_configs:
    catboost_metrics_by_split = {}
    catboost_preds_by_split = {}
    split_timer = ProgressTimer(total_steps=len(splits), label=f"Split evaluation [{scope_name}]")

    for split_idx, s in enumerate(splits, start=1):
        split_name = s["name"]
        logger.info(f"=== {split_name} | CatBoost recursive 16-day forecast | scope={scope_name} ===")

        model, feature_cols, cat_features_idx = fit_catboost_on_split(
            split=s,
            full_df=train_processed,
            top_pairs=top_pairs,
            use_top_pairs=use_top_pairs,
            best_params=best_params,
            thread_count=THREAD_COUNT,
        )

        metrics, val_pred_df = evaluate_catboost_on_split_recursive(
            split=s,
            full_df=train_processed,
            top_pairs=top_pairs,
            use_top_pairs=use_top_pairs,
            model=model,
            feature_cols=feature_cols,
            cat_features_idx=cat_features_idx,
            scope_name=scope_name,
        )

        catboost_metrics_by_split[split_name] = metrics
        catboost_preds_by_split[split_name] = val_pred_df
        split_timer.log_step(logger, f"{split_name} ({split_idx}/{len(splits)}), RMSLE={metrics['RMSLE']:.4f}")

    catboost_metrics_results[scope_name] = catboost_metrics_by_split
    catboost_preds_results[scope_name] = catboost_preds_by_split


2026-04-19 18:22:08,609 - catboost_store_level - INFO - === split_1 | CatBoost recursive 16-day forecast | scope=all_series ===
2026-04-19 18:22:08,610 - catboost_store_level - INFO - [split_1] Fitting CatBoost (train only) | scope=all_series
2026-04-19 18:22:08,611 - catboost_store_level - INFO - Preparing data for split_1 | scope=all_series
2026-04-19 18:22:10,660 - catboost_store_level - INFO - split_1 | scope=all_series | train rows after filtering: 2979504, val rows: 28512
2026-04-19 18:22:23,480 - catboost_store_level - INFO - split_1 | after lag/rolling dropna: train=2929608, val=28512
2026-04-19 18:22:29,443 - catboost_store_level - INFO - split_1 | X_train shape=(2929608, 25), X_val shape=(28512, 25), n_cat_features=12
2026-04-19 18:22:45,657 - catboost_store_level - INFO - [split_1] CatBoost fit started at 2026-04-19 18:22:45
2026-04-19 18:27:41,511 - catboost_store_level - INFO - [split_1] CatBoost fit finished in 295.9 seconds
2026-04-19 18:27:41,572 - catboost_store_level 

In [20]:
import pickle

catboost_metrics_tables = {}

for scope_name, catboost_metrics_by_split in catboost_metrics_results.items():
    rows = []
    for split_name, m in catboost_metrics_by_split.items():
        rows.append({
            'split': split_name,
            'model': m.get('model', 'catboost_store_family_recursive'),
            'scope': m.get('scope', scope_name),
            'RMSLE': m['RMSLE'],
            'RMSE': m['RMSE'],
            'MAE': m['MAE'],
            'WAPE': m['WAPE'],
        })

    catboost_metrics_df = pd.DataFrame(rows).set_index('split')
    metric_cols = ['RMSLE', 'RMSE', 'MAE', 'WAPE']
    mean_row = {col: catboost_metrics_df[col].mean() for col in metric_cols}
    mean_row['model'] = ''
    mean_row['scope'] = scope_name
    mean_row = pd.DataFrame(mean_row, index=['mean']).reindex(columns=catboost_metrics_df.columns)
    catboost_metrics_df = pd.concat([catboost_metrics_df, mean_row])
    catboost_metrics_tables[scope_name] = catboost_metrics_df

    metrics_path = METRICS_DIR / f'catboost_store_metrics_{scope_name}_recursive.csv'
    catboost_metrics_df.to_csv(metrics_path, index=True, index_label='split')
    logger.info(f'CatBoost recursive-forecast metrics saved to {metrics_path}')

    preds_path = PREDICTIONS_DIR / f'catboost_store_preds_{scope_name}_recursive.pkl'
    with open(preds_path, 'wb') as f:
        pickle.dump(catboost_preds_results[scope_name], f)
    logger.info(f'CatBoost predictions by split saved to {preds_path}')

    if scope_name == 'top_pairs':
        legacy_metrics_path = METRICS_DIR / 'catboost_store_metrics_by_split_recursive.csv'
        catboost_metrics_df.to_csv(legacy_metrics_path, index=True, index_label='split')
        legacy_preds_path = PREDICTIONS_DIR / 'catboost_store_preds_by_split_recursive.pkl'
        with open(legacy_preds_path, 'wb') as f:
            pickle.dump(catboost_preds_results[scope_name], f)
        logger.info(f'Legacy top_pairs CatBoost artifacts also saved to {legacy_metrics_path} and {legacy_preds_path}')

catboost_metrics_tables['all_series']


2026-04-19 18:52:55,685 - catboost_store_level - INFO - CatBoost recursive-forecast metrics saved to /home/jupyter/project/demand_forecasting/artifacts/metrics/catboost_store_metrics_all_series_recursive.csv
2026-04-19 18:52:55,769 - catboost_store_level - INFO - CatBoost predictions by split saved to /home/jupyter/project/demand_forecasting/artifacts/predictions/catboost_store_preds_all_series_recursive.pkl
2026-04-19 18:52:55,780 - catboost_store_level - INFO - CatBoost recursive-forecast metrics saved to /home/jupyter/project/demand_forecasting/artifacts/metrics/catboost_store_metrics_top_pairs_recursive.csv
2026-04-19 18:52:55,801 - catboost_store_level - INFO - CatBoost predictions by split saved to /home/jupyter/project/demand_forecasting/artifacts/predictions/catboost_store_preds_top_pairs_recursive.pkl
2026-04-19 18:52:55,829 - catboost_store_level - INFO - Legacy top_pairs CatBoost artifacts also saved to /home/jupyter/project/demand_forecasting/artifacts/metrics/catboost_stor

,model,scope,RMSLE,RMSE,MAE,WAPE
split_1,catboost_store_family_recursive,all_series,0.409149,241.242236,69.969306,0.149781
split_2,catboost_store_family_recursive,all_series,0.395645,270.631334,66.277115,0.130689
split_3,catboost_store_family_recursive,all_series,0.409635,326.826507,72.041587,0.144169
mean,,all_series,0.404809,279.566692,69.429336,0.141546


## Полное обучение и финальный прогноз

После завершения оценки на временных сплитах модель переобучается
на полном доступном обучающем периоде и используется для построения внешнего прогноза
на тестовом горизонте Kaggle.
Этот блок уже не предназначен для сравнительной оценки качества,
а представляет прикладной inference-этап на всей обучающей выборке.


In [21]:
def fill_na_in_cat(df, feature_cols, cat_features_idx):
    """
    Заполняет NaN в категориальных колонках специальным токеном "__MISSING__".
    Это нужно, поскольку CatBoost не принимает NaN в cat_features.
    """
    df = df.copy()
    cat_cols = [feature_cols[i] for i in cat_features_idx]
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype("object").fillna("__MISSING__")
    return df


In [22]:
def make_full_train_dataset(
    train_df,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Готовит полный датасет для обучения финальной CatBoost-модели
    и возвращает X_full, y_full, feature_cols, cat_features_idx, dates_aligned.
    """
    # Сортируем, чтобы лаги считались корректно
    df = train_df.sort_values(["store_nbr", "family", "date"]).copy()

    # Временные фичи
    df = add_time_features(df)

    # Лаги и rolling по sales
    df = add_lag_features(
        df,
        lags=lags,
        group_cols=["store_nbr", "family"],
        target_col="sales",
    )
    df = add_rolling_features(
        df,
        windows=roll_windows,
        group_cols=["store_nbr", "family"],
        target_col="sales",
    )

    # Все lag/rolling-фичи
    feature_cols_lag_roll = [
        c for c in df.columns
        if "lag_" in c or "rolling_mean_" in c
    ]

    # Отбрасываем строки без всех лагов/rolling
    df = df.dropna(subset=feature_cols_lag_roll)

    # Цель
    y_full = df["sales"].astype("float32")

    # Полный список фичей
    feature_cols = (
        NUM_FEATURES_BASE
        + TIME_FEATURES
        + feature_cols_lag_roll
        + BIN_FEATURES
        + CAT_FEATURES
    )
    feature_cols = [c for c in feature_cols if c in df.columns]

    X_full = df[feature_cols].copy()

    cat_features_idx = [
        i for i, c in enumerate(feature_cols)
        if c in (CAT_FEATURES + BIN_FEATURES)
    ]

    # Даты в том же порядке, что X_full / y_full
    dates_aligned = df["date"].values

    logger.info(
        f"[FULL] Dataset: X_full.shape={X_full.shape}, y_full.len={len(y_full)}, "
        f"n_cat_features={len(cat_features_idx)}"
    )

    return X_full, y_full, feature_cols, cat_features_idx, dates_aligned


In [23]:
# Строим полный датасет
X_full, y_full, feature_cols_full, cat_features_idx_full, dates_full = make_full_train_dataset(
    train_processed,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
)

# Последняя дата (валидация на один день)
last_date = dates_full.max()
logger.info(f"[FULL] Validation date = {pd.to_datetime(last_date).date()}")

val_mask = dates_full == last_date
train_mask = dates_full < last_date

X_train_full = X_full[train_mask]
y_train_full = y_full[train_mask]

X_val_full = X_full[val_mask]
y_val_full = y_full[val_mask]

logger.info(
    f"[FULL] Train size = {X_train_full.shape}, val size (1 day) = {X_val_full.shape}"
)


2026-04-19 18:53:37,988 - catboost_store_level - INFO - [FULL] Dataset: X_full.shape=(2958120, 25), y_full.len=2958120, n_cat_features=12
2026-04-19 18:53:38,066 - catboost_store_level - INFO - [FULL] Validation date = 2017-08-15
2026-04-19 18:53:38,515 - catboost_store_level - INFO - [FULL] Train size = (2956338, 25), val size (1 day) = (1782, 25)


In [24]:
THREAD_COUNT = -1  

# Обрабатываем NaN в категориальных фичах
X_train_full_filled = fill_na_in_cat(X_train_full, feature_cols_full, cat_features_idx_full)
X_val_full_filled   = fill_na_in_cat(X_val_full, feature_cols_full, cat_features_idx_full)

y_train_log = np.log1p(y_train_full)
y_val_log   = np.log1p(y_val_full)

train_pool_full = Pool(
    data=X_train_full_filled,
    label=y_train_log,
    cat_features=cat_features_idx_full,
)
val_pool_full = Pool(
    data=X_val_full_filled,
    label=y_val_log,
    cat_features=cat_features_idx_full,
)

final_model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    depth=best_params["depth"],
    learning_rate=best_params["learning_rate"],
    l2_leaf_reg=best_params["l2_leaf_reg"],
    bagging_temperature=best_params["bagging_temperature"],
    border_count=best_params["border_count"],
    random_state=42,
    thread_count=THREAD_COUNT,
    task_type=CATBOOST_TASK_TYPE,
    devices=CATBOOST_DEVICES if CATBOOST_TASK_TYPE == "GPU" else None,
    verbose=False,
    od_type="Iter",
    od_wait=50,
    iterations=5000,
)

start_time = datetime.now()
logger.info(f"[FULL] Final CatBoost fit started at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

final_model.fit(
    train_pool_full,
    eval_set=val_pool_full,
    use_best_model=True,
    verbose=False,
)

end_time = datetime.now()
elapsed = (end_time - start_time).total_seconds()
logger.info(f"[FULL] Final CatBoost fit finished in {elapsed:.1f} seconds")


2026-04-19 18:54:06,494 - catboost_store_level - INFO - [FULL] Final CatBoost fit started at 2026-04-19 18:54:06
2026-04-19 18:55:15,413 - catboost_store_level - INFO - [FULL] Final CatBoost fit finished in 68.9 seconds


In [25]:
def recursive_forecast_catboost_on_test(
    model,
    train_df,
    test_df,
    feature_cols,
    cat_features_idx,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Рекурсивный 16-дневный прогноз на тесте:
    - история = весь train (с реальными sales, включая последний день);
    - для каждого дня теста:
        * добавляем его каркас (без sales) к истории,
        * пересчитываем фичи (time, lag, rolling),
        * предсказываем sales,
        * подставляем предсказания в историю, чтобы они участвовали в лагах
          на следующих днях;
    - в конце мержим прогнозы с test_df по (store_nbr, family, date),
      чтобы вернуть id и получить submission формата id,sales.
    """
    # История: весь train
    hist = train_df.sort_values(["store_nbr", "family", "date"]).copy()

    # Тест: каркас (есть id, store_nbr, family, date и все exogenous-признаки)
    future = test_df.sort_values(["date", "store_nbr", "family"]).copy()

    test_dates = sorted(future["date"].unique())
    all_preds = []
    forecast_timer = ProgressTimer(total_steps=len(test_dates), label="[TEST] Recursive forecast")

    for cur_date in test_dates:
        cur_ts = pd.to_datetime(cur_date)
        logger.info(f"[TEST] Forecasting date {cur_ts.date()}")

        # Строки теста для текущей даты
        day_future = future[future["date"] == cur_date].copy()

        # Временный столбец sales=NaN, чтобы лаги считались только из hist
        day_future["sales"] = np.nan

        # История + текущий день
        combined = pd.concat([hist, day_future], axis=0, ignore_index=True)

        # Фичи времени + lag/rolling по sales
        combined = add_time_features(combined)
        combined = add_lag_features(
            combined,
            lags=lags,
            group_cols=["store_nbr", "family"],
            target_col="sales",
        )
        combined = add_rolling_features(
            combined,
            windows=roll_windows,
            group_cols=["store_nbr", "family"],
            target_col="sales",
        )

        # Берём только строки текущего дня уже с фичами
        cur_df = combined[combined["date"] == cur_date].copy()

        # ВАЖНО: на тесте НЕ делаем dropna по lag/rolling — CatBoost умеет с ними работать
        # cur_df = cur_df.dropna(subset=feature_cols_lag_roll)

        if cur_df.empty:
            logger.warning(f"[TEST] No rows to predict for {cur_ts.date()}, skipping")
            forecast_timer.log_step(logger, f"date={cur_ts.date()} skipped")
            continue

        # X в том же порядке фичей, что при обучении
        X_cur = cur_df[feature_cols].copy()
        X_cur = fill_na_in_cat(X_cur, feature_cols, cat_features_idx)

        pool_cur = Pool(
            data=X_cur,
            cat_features=cat_features_idx,
        )

        # Предсказания в лог-пространстве → исходный масштаб
        y_pred_log = model.predict(pool_cur)
        y_pred = np.expm1(y_pred_log)
        y_pred = np.clip(y_pred, 0, None)

        # Сохраняем прогнозы вместе с ключами
        cur_df = cur_df[["store_nbr", "family", "date"]].copy()
        cur_df["sales"] = y_pred
        all_preds.append(cur_df)

        # Обновляем историю: на cur_date sales = предсказания (для будущих лагов)
        hist_update = day_future.copy()
        hist_update["sales"] = y_pred
        hist = pd.concat([hist, hist_update], axis=0, ignore_index=True)
        forecast_timer.log_step(logger, f"date={cur_ts.date()}")

    # Собираем все прогнозы по ключам
    preds_df = pd.concat(all_preds, axis=0, ignore_index=True)

    # Мерджим с test_df, чтобы вернуть id для сабмита
    submission_df = (
        test_df[["id", "store_nbr", "family", "date"]]
        .merge(
            preds_df,
            on=["store_nbr", "family", "date"],
            how="left",
        )
    )

    # На всякий случай проверим покрытие id
    missing_ids = set(test_df["id"]) - set(submission_df["id"])
    if missing_ids:
        logger.warning(f"[TEST] Missing predictions for {len(missing_ids)} ids")

    # Финальный формат для Kaggle
    submission_df = submission_df[["id", "sales"]]

    return submission_df


In [26]:
submission_df = recursive_forecast_catboost_on_test(
    model=final_model,
    train_df=train_processed,
    test_df=test_processed,
    feature_cols=feature_cols_full,
    cat_features_idx=cat_features_idx_full,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
)
submission = submission_df.sort_values('id')[['id', 'sales']]
submission_path = SUBMISSIONS_DIR / 'submission_catboost_recursive.csv'
submission.to_csv(submission_path, index=False)
logger.info(f'CatBoost submission saved to {submission_path}')
submission.head()


2026-04-19 18:55:17,062 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-16
2026-04-19 18:55:25,413 - catboost_store_level - INFO - [TEST] Recursive forecast | done 1/16 (date=2017-08-16) | elapsed=8s | eta=2m 05s
2026-04-19 18:55:25,414 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-17
2026-04-19 18:55:34,280 - catboost_store_level - INFO - [TEST] Recursive forecast | done 2/16 (date=2017-08-17) | elapsed=17s | eta=2m 00s
2026-04-19 18:55:34,281 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-18
2026-04-19 18:55:42,186 - catboost_store_level - INFO - [TEST] Recursive forecast | done 3/16 (date=2017-08-18) | elapsed=25s | eta=1m 48s
2026-04-19 18:55:42,187 - catboost_store_level - INFO - [TEST] Forecasting date 2017-08-19
2026-04-19 18:55:50,207 - catboost_store_level - INFO - [TEST] Recursive forecast | done 4/16 (date=2017-08-19) | elapsed=33s | eta=1m 39s
2026-04-19 18:55:50,208 - catboost_store_level - INFO - [TEST] Forecasting date 2

,id,sales
0,3000888,4.350312
16,3000889,0.045625
32,3000890,4.218325
48,3000891,2254.204322
64,3000892,0.045912
